# Lab 6: Trees, Forests and Boosting

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

{{COLAB_BADGE}}

This lab accompanies Lecture 6. By the end of the session you should be able to:

1. Grow a regression tree from scratch by greedy splitting, and see why the algorithm is myopic.
2. Demonstrate that single trees are high-variance estimators, and reduce that variance by bagging.
3. Explain what randomising the split candidates adds beyond bootstrapping, and use out-of-bag error to tune a forest for free.
4. Implement gradient boosting with stumps and describe how it differs from bagging in one sentence.
5. Interpret a fitted ensemble with permutation importance and partial dependence, and state honestly what those tools do and do not tell you.
6. Compare tree ensembles against the penalised linear models of Lab 3 on the same forecasting problem.

**Plan for the session**

| Time | Part |
|---|---|
| 0:00 - 0:25 | Part 1. Growing a tree |
| 0:25 - 0:45 | Part 2. Instability and bagging |
| 0:45 - 1:05 | Part 3. Random forests |
| 1:05 - 1:30 | Part 4. Boosting |
| 1:30 - 1:45 | Part 5. Interpretation |
| 1:45 - 2:00 | Part 6. Forecasting horse race |

## Setup

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/korobilis/ECON5129-labs/main"

if not os.path.exists("econ5129_utils.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/econ5129_utils.py", "econ5129_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import econ5129_utils as e5

e5.set_style()
rng = np.random.default_rng(5129)

## Part 1. Growing a tree

A regression tree partitions the predictor space into rectangles and predicts the mean of the response within each. Finding the optimal partition is computationally infeasible, so the tree is grown greedily: at each node, search over every variable and every possible split point for the split that reduces the sum of squared errors most, then recurse.

The whole algorithm is about thirty lines.

In [ ]:
def best_split(X, y):
    """Find the variable and threshold that most reduce the sum of squared errors."""
    n, p = X.shape
    best = {"score": np.sum((y - y.mean()) ** 2), "feature": None, "threshold": None}

    for j in range(p):
        order = np.argsort(X[:, j])
        x_sorted, y_sorted = X[order, j], y[order]

        cumsum = np.cumsum(y_sorted)
        cumsq = np.cumsum(y_sorted ** 2)
        total, total_sq = cumsum[-1], cumsq[-1]

        for i in range(1, n):
            if x_sorted[i] == x_sorted[i - 1]:
                continue
            left_n, right_n = i, n - i
            sse_left = cumsq[i - 1] - cumsum[i - 1] ** 2 / left_n
            sse_right = (total_sq - cumsq[i - 1]) - (total - cumsum[i - 1]) ** 2 / right_n
            score = sse_left + sse_right
            if score < best["score"]:
                best = {"score": score, "feature": j,
                        "threshold": 0.5 * (x_sorted[i] + x_sorted[i - 1])}
    return best


def grow_tree(X, y, depth=0, max_depth=3, min_samples=10):
    """Recursively grow a regression tree."""
    node = {"value": float(y.mean()), "n": len(y)}
    if depth >= max_depth or len(y) < 2 * min_samples:
        return node

    split = best_split(X, y)
    if split["feature"] is None:
        return node

    mask = X[:, split["feature"]] <= split["threshold"]
    if mask.sum() < min_samples or (~mask).sum() < min_samples:
        return node

    node["feature"] = split["feature"]
    node["threshold"] = split["threshold"]
    node["left"] = grow_tree(X[mask], y[mask], depth + 1, max_depth, min_samples)
    node["right"] = grow_tree(X[~mask], y[~mask], depth + 1, max_depth, min_samples)
    return node


def predict_tree(node, X):
    """Predict by walking each observation down the tree."""
    if "feature" not in node:
        return np.full(len(X), node["value"])

    out = np.empty(len(X))
    mask = X[:, node["feature"]] <= node["threshold"]
    if mask.any():
        out[mask] = predict_tree(node["left"], X[mask])
    if (~mask).any():
        out[~mask] = predict_tree(node["right"], X[~mask])
    return out

The data generating process below has a threshold in it: the slope changes sign at $x = 0$. This is the kind of structure a tree finds easily and a linear model cannot represent at all.

In [ ]:
n = 300
x_1d = np.sort(rng.uniform(-3, 3, n))
y_1d = np.where(x_1d < 0, 1.0 + 0.5 * x_1d, 2.0 - 1.2 * x_1d) + 0.4 * rng.normal(size=n)
X_1d = x_1d[:, None]

grid = np.linspace(-3, 3, 400)[:, None]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, depth in zip(axes, [1, 2, 5]):
    tree = grow_tree(X_1d, y_1d, max_depth=depth)
    ax.scatter(x_1d, y_1d, s=12, color=e5.COLORS[6], alpha=0.5)
    ax.plot(grid.ravel(), predict_tree(tree, grid), color=e5.COLORS[2], lw=2)
    ax.set_title(f"max depth {depth}")
fig.suptitle("A tree approximates any function by a step function")
fig.tight_layout()
plt.show()

Check the implementation against the reference one, and note how much faster the reference is: `scikit-learn` implements the same algorithm in compiled code.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

sk_tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=10).fit(X_1d, y_1d)
our_tree = grow_tree(X_1d, y_1d, max_depth=2)

print(f"our first split      : x <= {our_tree['threshold']:.4f}")
print(f"scikit-learn's split : x <= {sk_tree.tree_.threshold[0]:.4f}")
print(f"maximum difference in predictions: "
      f"{np.max(np.abs(predict_tree(our_tree, grid) - sk_tree.predict(grid))):.4f}")

## Part 2. Instability and bagging

A greedy tree commits to its first split before seeing what happens later. Change the sample slightly and that split can move, taking the entire structure below it with it. Trees are therefore low-bias, high-variance estimators, which is exactly the profile that averaging repairs.

In [ ]:
fig, ax = plt.subplots()
for _ in range(12):
    idx = rng.choice(n, size=n, replace=True)
    tree_b = DecisionTreeRegressor(max_depth=4, min_samples_leaf=5).fit(X_1d[idx], y_1d[idx])
    ax.plot(grid.ravel(), tree_b.predict(grid), color=e5.COLORS[6], lw=0.8, alpha=0.6)
ax.scatter(x_1d, y_1d, s=10, color=e5.COLORS[1], alpha=0.4)
ax.set_title("Twelve trees, twelve bootstrap samples, twelve different answers")
plt.show()

**Bagging** averages trees grown on bootstrap resamples. Because the bootstrap samples are drawn from the same data the trees are correlated, so the variance falls but not by the factor $1/B$ that independence would give.

In [ ]:
def bagged_trees(X, y, X_eval, n_trees=200, max_depth=6, generator=None):
    """Average of trees fitted to bootstrap resamples."""
    generator = generator or np.random.default_rng(0)
    preds = np.empty((n_trees, len(X_eval)))
    for b in range(n_trees):
        idx = generator.choice(len(y), size=len(y), replace=True)
        tree = DecisionTreeRegressor(max_depth=max_depth, min_samples_leaf=5)
        tree.fit(X[idx], y[idx])
        preds[b] = tree.predict(X_eval)
    return preds


preds = bagged_trees(X_1d, y_1d, grid, n_trees=200, generator=rng)

fig, ax = plt.subplots()
ax.scatter(x_1d, y_1d, s=10, color=e5.COLORS[6], alpha=0.4)
single = DecisionTreeRegressor(max_depth=6, min_samples_leaf=5).fit(X_1d, y_1d)
ax.plot(grid.ravel(), single.predict(grid), color=e5.COLORS[6], lw=1.0, label="single tree")
ax.plot(grid.ravel(), preds.mean(axis=0), color=e5.COLORS[0], lw=2.2, label="bagged, 200 trees")
ax.set_title("Averaging smooths the steps without changing the shape")
ax.legend()
plt.show()

### Exercise 1

Quantify the variance reduction rather than admiring the picture.

1. Simulate 50 independent datasets from the same process.
2. For each, fit a single depth-6 tree and a bagged ensemble of 50 trees, and record the prediction at $x_0 = 1.0$.
3. Report the bias, variance and mean squared error of both estimators at that point, using the true conditional mean $f(1.0)$.

By what factor does bagging reduce the variance, and what happens to the bias?

In [ ]:
def f_true_1d(x):  #@keep
    return np.where(x < 0, 1.0 + 0.5 * x, 2.0 - 1.2 * x)  #@keep


x0 = np.array([[1.0]])  #@keep
target = f_true_1d(1.0)  #@keep

single_preds, bagged_preds = [], []
for _ in range(50):
    xs = np.sort(rng.uniform(-3, 3, n))
    ys = f_true_1d(xs) + 0.4 * rng.normal(size=n)
    Xs = xs[:, None]

    single_preds.append(DecisionTreeRegressor(max_depth=6, min_samples_leaf=5)
                        .fit(Xs, ys).predict(x0)[0])
    bagged_preds.append(bagged_trees(Xs, ys, x0, n_trees=50, generator=rng).mean())

summary = pd.DataFrame({
    "estimator": ["single tree", "bagged (50 trees)"],
    "bias": [np.mean(single_preds) - target, np.mean(bagged_preds) - target],
    "variance": [np.var(single_preds), np.var(bagged_preds)],
}).set_index("estimator")
summary["MSE"] = summary["bias"] ** 2 + summary["variance"]
print(summary.round(5))
print(f"\nvariance reduction factor: {np.var(single_preds) / np.var(bagged_preds):.2f}")

## Part 3. Random forests

Bagged trees remain correlated because a dominant predictor is chosen for the first split in nearly every bootstrap sample. Random forests break that correlation by offering each split a random subset of `max_features` predictors, so weaker predictors sometimes get their turn. Averaging correlated trees helps; averaging less correlated trees helps more.

Forests also come with a free validation set. Each bootstrap sample omits about 37% of the observations, and averaging the predictions of the trees that did not see observation $i$ gives an out-of-bag estimate of test error.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_multi = rng.normal(size=(500, 8))
y_multi = (2.0 * np.sin(np.pi * X_multi[:, 0])
           + X_multi[:, 1] * X_multi[:, 2]
           + 0.5 * X_multi[:, 3] ** 2
           + rng.normal(scale=0.5, size=500))

rows = []
for mf in [1, 2, 3, 5, 8]:
    forest = RandomForestRegressor(n_estimators=300, max_features=mf, oob_score=True,
                                   random_state=0, n_jobs=1).fit(X_multi, y_multi)
    rows.append({"max_features": mf, "out-of-bag $R^2$": forest.oob_score_})

print(pd.DataFrame(rows).set_index("max_features").round(4))
print("\nmax_features = 8 is bagging: every split sees every predictor.")

## Part 4. Boosting

Bagging fits many deep trees independently and averages them, attacking variance. Boosting fits many shallow trees sequentially, each one to the residuals left by its predecessors, attacking bias.

With squared error loss the algorithm is short: start from the mean, repeatedly fit a stump to the current residuals, and add a small multiple $\nu$ of it to the running prediction.

In [ ]:
def gradient_boost(X, y, X_eval, n_rounds=300, learning_rate=0.1, max_depth=2):
    """Gradient boosting with squared error loss."""
    prediction = np.full(len(y), y.mean())
    evaluation = np.full(len(X_eval), y.mean())
    staged = []

    for _ in range(n_rounds):
        residual = y - prediction
        stump = DecisionTreeRegressor(max_depth=max_depth).fit(X, residual)
        prediction += learning_rate * stump.predict(X)
        evaluation += learning_rate * stump.predict(X_eval)
        staged.append(evaluation.copy())

    return evaluation, np.array(staged)


final, staged = gradient_boost(X_1d, y_1d, grid, n_rounds=300, learning_rate=0.1)

fig, ax = plt.subplots()
ax.scatter(x_1d, y_1d, s=10, color=e5.COLORS[6], alpha=0.4)
for rounds, colour in zip([1, 10, 50, 300], [e5.COLORS[3], e5.COLORS[4], e5.COLORS[1], e5.COLORS[0]]):
    ax.plot(grid.ravel(), staged[rounds - 1], color=colour, label=f"{rounds} rounds")
ax.set_title("Boosting builds the fit up in small steps")
ax.legend(fontsize=9)
plt.show()

### Exercise 2

Boosting overfits if it is run for too long. The learning rate and the number of rounds trade off against each other.

1. Split the simulated data into training and test halves.
2. For learning rates 0.01, 0.1 and 0.5, compute test error after each boosting round up to 500 rounds.
3. Plot the three curves and identify where each starts to overfit.

Which combination of learning rate and number of rounds would you choose, and what is the cost of a small learning rate?

In [ ]:
split_at = 150  #@keep
Xb_tr, Xb_te = X_1d[:split_at], X_1d[split_at:]  #@keep
yb_tr, yb_te = y_1d[:split_at], y_1d[split_at:]  #@keep

fig, ax = plt.subplots()
for lr in [0.01, 0.1, 0.5]:
    _, staged_te = gradient_boost(Xb_tr, yb_tr, Xb_te, n_rounds=500, learning_rate=lr)
    errors = [e5.mse(yb_te, s) for s in staged_te]
    ax.plot(errors, label=f"learning rate {lr}, best at round {int(np.argmin(errors)) + 1}")
ax.set_xlabel("boosting round")
ax.set_ylabel("test mean squared error")
ax.set_title("Small steps take longer but stop in a better place")
ax.legend(fontsize=9)
plt.show()

## Part 5. Interpretation

An ensemble of several hundred trees has no coefficients to report. Two tools fill the gap, and both must be read carefully.

**Permutation importance** shuffles one predictor and measures how much accuracy falls. It answers "how much does the fitted model rely on this variable", not "how important is this variable in the world".

**Partial dependence** averages the fitted function over the empirical distribution of the other predictors. It is a causal object only if the model is causal, which it is not.

In [ ]:
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.model_selection import train_test_split

Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_multi, y_multi, test_size=0.3, random_state=0)
forest = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=1).fit(Xm_tr, ym_tr)

imp = permutation_importance(forest, Xm_te, ym_te, n_repeats=10, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = np.argsort(imp.importances_mean)
axes[0].barh([f"$x_{{{j + 1}}}$" for j in order], imp.importances_mean[order],
             xerr=imp.importances_std[order], color=e5.COLORS[1])
axes[0].set_xlabel("drop in $R^2$ when shuffled")
axes[0].set_title("Permutation importance")

pd_result = partial_dependence(forest, Xm_tr, features=[0], grid_resolution=50)
axes[1].plot(pd_result["grid_values"][0], pd_result["average"][0], color=e5.COLORS[0])
axes[1].set_xlabel("$x_1$")
axes[1].set_ylabel("partial dependence")
axes[1].set_title("Partial dependence on $x_1$, true effect is $2\\sin(\\pi x_1)$")
fig.tight_layout()
plt.show()

The forest recovers the shape of the sine without being told that a sine exists, and it correctly identifies that predictors five to eight do nothing. Note also that $x_2$ and $x_3$ matter only through their interaction, which permutation importance registers but partial dependence on either one alone would hide.

## Part 6. Forecasting horse race

Back to the problem from Lab 3: the monthly change in unemployment, one month ahead, from 363 macroeconomic predictors.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import TimeSeriesSplit

data, codes = e5.load_fredmd()
dataset = e5.build_forecast_dataset(data, codes, target="UNRATE", horizon=1, n_lags=3)
X, y, dates = dataset["X"], dataset["y"], dataset["dates"]

split = int(0.7 * len(y))
X_tr, X_te, y_tr, y_te = X[:split], X[split:], y[:split], y[split:]
Z_tr, Z_te, _, _ = e5.standardise(X_tr, X_te)
y_mean = y_tr.mean()

ar_cols = [i for i, name in enumerate(dataset["names"]) if name.startswith("UNRATE.")]
b_ar, *_ = np.linalg.lstsq(np.column_stack([np.ones(split), X_tr[:, ar_cols]]), y_tr, rcond=None)
bench = np.column_stack([np.ones(len(y_te)), X_te[:, ar_cols]]) @ b_ar

tscv = TimeSeriesSplit(n_splits=5)
models = {
    "LASSO": LassoCV(alphas=np.logspace(-4, 1, 60), cv=tscv, max_iter=50000),
    "ridge": RidgeCV(alphas=np.logspace(0, 7, 50), cv=tscv),
    "random forest": RandomForestRegressor(n_estimators=400, max_features=0.3,
                                           min_samples_leaf=5, random_state=0, n_jobs=1),
    "gradient boosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05,
                                                       max_depth=3, random_state=0),
}

rows = []
predictions = {}
for name, model in models.items():
    model.fit(Z_tr, y_tr - y_mean)
    pred = model.predict(Z_te) + y_mean
    predictions[name] = pred
    rows.append({"model": name, "RMSE": e5.rmse(y_te, pred),
                 "out-of-sample $R^2$": e5.oos_r2(y_te, pred, benchmark=bench)})

rows.append({"model": "AR(3) benchmark", "RMSE": e5.rmse(y_te, bench), "out-of-sample $R^2$": 0.0})
print(pd.DataFrame(rows).set_index("model").round(4))

### Exercise 3

Averages hide the interesting part. Nonlinear methods are usually claimed to earn their keep in turbulent periods rather than on average.

1. Split the evaluation sample into recession and expansion months using `e5.load_usrec()`.
2. Report the root mean squared error of each model separately for the two regimes.
3. Test whether the best nonlinear model beats the LASSO using `e5.diebold_mariano`.

Does the ranking of methods depend on the state of the economy?

In [ ]:
recession = e5.load_usrec().reindex(dates[split:]).to_numpy() == 1.0  #@keep

rows = []
for name, pred in predictions.items():
    rows.append({
        "model": name,
        "RMSE, recessions": e5.rmse(y_te[recession], pred[recession]),
        "RMSE, expansions": e5.rmse(y_te[~recession], pred[~recession]),
    })
rows.append({"model": "AR(3) benchmark",
             "RMSE, recessions": e5.rmse(y_te[recession], bench[recession]),
             "RMSE, expansions": e5.rmse(y_te[~recession], bench[~recession])})

print(pd.DataFrame(rows).set_index("model").round(4))
print(f"\nrecession months in the evaluation sample: {recession.sum()} of {len(recession)}")

stat, pval = e5.diebold_mariano(y_te - predictions["gradient boosting"],
                                y_te - predictions["LASSO"])
print(f"\nDiebold-Mariano, boosting against LASSO: statistic {stat:.3f}, p-value {pval:.3f}")
print("A negative statistic favours boosting.")

### Exercise 4

Which predictors does the forest use, and does it agree with the LASSO?

1. Compute permutation importance for the fitted random forest on the evaluation sample. Use `n_repeats=5` to keep it quick.
2. List the ten most important predictors.
3. Compare with the predictors the LASSO selected. Do the two methods identify the same series?

In [ ]:
forest_macro = models["random forest"]  #@keep
imp_macro = permutation_importance(forest_macro, Z_te, y_te - y_mean,  #@keep
                                   n_repeats=5, random_state=0)  #@keep

order = np.argsort(-imp_macro.importances_mean)[:10]
forest_top = [dataset["names"][i] for i in order]

lasso_coef = models["LASSO"].coef_
lasso_order = np.argsort(-np.abs(lasso_coef))[:10]
lasso_top = [dataset["names"][i] for i in lasso_order if lasso_coef[i] != 0]

print(pd.DataFrame({
    "forest, by permutation importance": forest_top,
    "LASSO, by absolute coefficient": lasso_top + [""] * (10 - len(lasso_top)),
}).to_string(index=False))

print(f"\nseries appearing in both lists: {sorted(set(forest_top) & set(lasso_top))}")

## Take-home challenges

1. **Trees cannot extrapolate.** Fit a forest to data on $x \in [-3, 3]$ and predict at $x = 5$. Compare with a linear model. Explain the result in terms of how a tree makes predictions, and say what this implies for forecasting a trending series.

2. **Hybrid models.** Estimate the effect of a treatment on an outcome when both depend on a high-dimensional set of controls, using the partialling-out approach: predict the outcome from the controls with a forest, predict the treatment from the controls with a forest, and regress one set of residuals on the other. Compare with ordinary least squares including all controls linearly.

3. **Honest importance.** Permutation importance computed on training data is biased towards variables with many distinct values. Demonstrate this by adding a pure noise predictor drawn from a continuous distribution and another taking three values, then compare importances computed in sample and out of sample.

4. **Ensembling across model classes.** Average the forecasts of the LASSO and the boosted trees with equal weights. Does the combination beat both of its components, and why is that so common?

---

**Next**: Lecture 7 replaces the fixed basis of Lab 5 and the partition of Lab 6 with a basis that is learned from the data. Lab 7 builds a neural network from scratch, including backpropagation.